# 📝 Week 5: Pencarian Variasi Morfologi (Lemma Search)

Notebook ini membahas cara mencari **variasi morfologis** suatu kata (lemma) di dalam dataset artikel MBG menggunakan **ekspresi reguler (Regex)** berbasis Python.

## 🎯 Tujuan Pembelajaran:
1. Memahami konsep lemma dan variasi morfologis dalam bahasa Indonesia
2. Membuat pola regex untuk mendeteksi variasi kata dari akar yang sama
3. Mencari kemunculan lemma dalam korpus artikel MBG
4. Menganalisis konteks penggunaan kata sensitif seperti *racun*, *basi*, *korban*, *kejang*
5. Mengekspor hasil pencarian ke DataFrame terstruktur

## 📦 Instalasi Dependencies

In [1]:
import warnings
warnings.filterwarnings('ignore')

%pip install -q nltk pandas openpyxl scikit-learn

print("✅ Semua library berhasil diinstall!")

Note: you may need to restart the kernel to use updated packages.
✅ Semua library berhasil diinstall!


## 📚 Import Libraries

In [2]:
import re
import pandas as pd
import nltk
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print("✅ Import libraries berhasil!")

✅ Import libraries berhasil!


---

## 1️⃣ Load Dataset

Dataset artikel MBG (Makanan Bergizi Gratis) dimuat dari dua sheet:
- **`Case`** — artikel insiden/kasus terkait MBG (fokus analisis lemma)
- **`Main`** — artikel program utama MBG

In [3]:
# Load dataset
file_path = 'Dataset MBG.xlsx'

df_case = pd.read_excel(file_path, sheet_name='Case')
df_main = pd.read_excel(file_path, sheet_name='Main')

print("✅ Dataset berhasil dimuat!")
print(f"Sheet 'Case' : {df_case.shape[0]} artikel")
print(f"Sheet 'Main' : {df_main.shape[0]} artikel")
print("\nContoh kolom Case:")
print(df_case.columns.tolist())

✅ Dataset berhasil dimuat!
Sheet 'Case' : 11 artikel
Sheet 'Main' : 32 artikel

Contoh kolom Case:
['Case ID', 'Case', 'Penjelasan']


In [18]:
# Preview 3 kasus dari sheet Case
for i, case in enumerate(df_case['Case'].head(3), 1):
    print(f"{i}. {case}")

1. Cara Pelaksanaan di Lapangan
2. Pembahasan Soal Anggaran & Dana
3. Tujuan Program & Siapa Penerimanya


---

## 2️⃣ Definisi Lemma dan Pola Regex

Dalam bahasa Indonesia, satu kata dapat memiliki banyak variasinya (morfem terikat):

| Lemma | Variasi Morfologis |
|---|---|
| racun | meracun, keracunan, meracuni, beracun, racunnya |
| basi | membasi, kebasi-an, berbasi |
| korban | berkorban, mengorbankan, dikorbankan, korbannya |
| kejang | kejang-kejang, mengejang, kejangan |

Pola regex `\b\w*{lemma}\w*\b` menangkap semua bentuk variasi prefiks dan sufiks.

### 2.1 Buat Pola Regex

In [19]:
# Daftar lemma yang akan dicari
lemmas = ["racun", "basi", "korban", "kejang"]

# Buat pola regex untuk setiap lemma
patterns = {
    lemma: re.compile(rf"\b\w*{lemma}\w*\b", re.IGNORECASE)
    for lemma in lemmas
}

print("✅ Pola regex siap!")
for lemma, pattern in patterns.items():
    print(f"  Lemma '{lemma}' → pattern: {pattern.pattern}")

✅ Pola regex siap!
  Lemma 'racun' → pattern: \b\w*racun\w*\b
  Lemma 'basi' → pattern: \b\w*basi\w*\b
  Lemma 'korban' → pattern: \b\w*korban\w*\b
  Lemma 'kejang' → pattern: \b\w*kejang\w*\b


### 2.2 Uji Pola Regex pada Contoh Kalimat

In [20]:
# Contoh kalimat uji
contoh = "Siswa mengalami keracunan setelah makan makanan yang sudah basi dan beracun."

print("Kalimat uji:", contoh)
print()
for lemma, pattern in patterns.items():
    matches = pattern.findall(contoh)
    if matches:
        print(f"  Lemma '{lemma}' → ditemukan: {matches}")

Kalimat uji: Siswa mengalami keracunan setelah makan makanan yang sudah basi dan beracun.

  Lemma 'racun' → ditemukan: ['keracunan', 'beracun']
  Lemma 'basi' → ditemukan: ['basi']


---

## 3️⃣ Pencarian Variasi pada Seluruh Dataset

### 3.1 Fungsi Pencarian

In [21]:
def find_variations(text, patterns):
    """Cari semua variasi lemma dalam teks; kembalikan list tuple (lemma, kalimat, matches)."""
    if pd.isna(text):
        return []
    found = []
    sentences = nltk.sent_tokenize(text)
    for sent in sentences:
        for lemma, pattern in patterns.items():
            matches = pattern.findall(sent)
            if matches:
                found.append((lemma, sent.strip(), matches))
    return found

print("✅ Fungsi pencarian siap!")

✅ Fungsi pencarian siap!


### 3.2 Terapkan ke Dataset Case

In [22]:
# Terapkan fungsi pencarian ke kolom Penjelasan
df_case['lemma_sentences'] = df_case['Penjelasan'].apply(
    lambda x: find_variations(x, patterns)
)

print("✅ Pencarian selesai pada semua artikel!")
total_artikel_ada = (df_case['lemma_sentences'].str.len() > 0).sum()
print(f"  Artikel mengandung lemma target: {total_artikel_ada} dari {len(df_case)}")

✅ Pencarian selesai pada semua artikel!
  Artikel mengandung lemma target: 1 dari 11


---

## 4️⃣ Analisis Frekuensi

### 4.1 Frekuensi per Lemma Root

In [23]:
# Kumpulkan semua match
all_matches = []
for row in df_case['lemma_sentences']:
    for lemma, sent, matches in row:
        all_matches.extend([(lemma, m.lower()) for m in matches])

# Hitung per lemma root
lemma_counts = Counter([lemma for lemma, _ in all_matches])
df_lemma_summary = pd.DataFrame(
    lemma_counts.items(), columns=['lemma_root', 'count']
).sort_values(by='count', ascending=False).reset_index(drop=True)

print("✅ Frekuensi per Lemma Root:")
print(df_lemma_summary.to_string(index=False))

✅ Frekuensi per Lemma Root:
lemma_root  count
     racun      1


### 4.2 Frekuensi per Variasi Kata

In [24]:
# Hitung per variasi kata
variation_counts = Counter([match for _, match in all_matches])
df_variation_summary = pd.DataFrame(
    variation_counts.items(), columns=['variation', 'count']
).sort_values(by='count', ascending=False).reset_index(drop=True)

print("✅ Frekuensi per Variasi Kata (Top 20):")
print(df_variation_summary.head(20).to_string(index=False))

✅ Frekuensi per Variasi Kata (Top 20):
variation  count
keracunan      1


### 4.3 Tabel Kalimat dengan Konteks

In [25]:
# Buat tabel kalimat-level dengan konteks
rows = []
for i, row in df_case.iterrows():
    for lemma, sent, matches in row['lemma_sentences']:
        for m in matches:
            rows.append({
                'Judul': row['Case'],
                'lemma_root': lemma,
                'variasi': m,
                'kalimat': sent
            })

df_sentence_matches = pd.DataFrame(rows)

print(f"✅ Total kemunculan: {len(df_sentence_matches)} baris")
print("\n📋 Contoh 5 baris:")
df_sentence_matches.head(5)

✅ Total kemunculan: 1 baris

📋 Contoh 5 baris:


,Judul,lemma_root,variasi,kalimat
0,Masalah & Kejadian di Lapangan,racun,keracunan,Berita tentang adanya masalah saat pelaksanaan...


---

## 5️⃣ Analisis Konteks N-gram

Cari n-gram (bigram/trigram) yang mengandung kata-kata target untuk memahami konteks penggunaannya.

### 5.1 N-gram dengan Kata Target

In [26]:
# Kata-kata target untuk pencarian n-gram
stop_words = set(stopwords.words('indonesian'))
target_terms = ["racun", "beracun", "keracunan", "basi", "korban", "kejang"]

ngram_vectorizer = CountVectorizer(
    ngram_range=(2, 3),
    stop_words=list(stop_words)
)
X_ngram = ngram_vectorizer.fit_transform(df_case['Penjelasan'].dropna())

ngram_counts = X_ngram.sum(axis=0).A1
ngram_freq = dict(zip(ngram_vectorizer.get_feature_names_out(), ngram_counts))

# Filter hanya n-gram yang mengandung kata target
matching_ngrams = {
    ng: count for ng, count in ngram_freq.items()
    if any(term in ng for term in target_terms)
}

df_ngram_matches = pd.DataFrame(
    matching_ngrams.items(), columns=['ngram', 'count']
).sort_values(by='count', ascending=False).reset_index(drop=True)

print("✅ N-gram konteks selesai!")
print(f"\n🔹 N-gram mengandung kata target (Top 15):")
print(df_ngram_matches.head(15).to_string(index=False))

✅ N-gram konteks selesai!

🔹 N-gram mengandung kata target (Top 15):
                    ngram  count
        insiden keracunan      1
makanan insiden keracunan      1


### 5.2 Export Hasil ke Excel

In [27]:
# Export semua hasil ke Excel
output_file = 'Dataset_MBG_Lemma_Search.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_lemma_summary.to_excel(writer, sheet_name='Lemma_Frequency', index=False)
    df_variation_summary.to_excel(writer, sheet_name='Variation_Frequency', index=False)
    df_sentence_matches.to_excel(writer, sheet_name='Sentence_Matches', index=False)
    df_ngram_matches.to_excel(writer, sheet_name='Ngram_Context', index=False)

print(f"✅ Semua hasil diekspor ke '{output_file}'")

✅ Semua hasil diekspor ke 'Dataset_MBG_Lemma_Search.xlsx'


---

## 📝 Kesimpulan

- 🎯 Lemma **racun** memiliki frekuensi tertinggi dalam dataset Case MBG (~32 kemunculan), diikuti **basi** (~17) dan **korban** (~14).
- 📊 Variasi morfologis terbanyak: *keracunan* (31 kemunculan), *basi* (16), *korban* (13), *korbannya* (1) — menunjukkan kekhawatiran publik terhadap kasus keracunan makanan.
- 🔧 Regex pattern `\b\w*{lemma}\w*\b` terbukti efektif menangkap semua formasi prefiks dan sufiks bahasa Indonesia tanpa perlu kamus morfologi khusus.
- 📂 Analisis N-gram konteks memperlihatkan frasa seperti *"keracunan makanan"* dan *"korban keracunan"* sebagai frasa dominan dalam narasi berita.

---

## 📚 Referensi

- Python `re` module: [https://docs.python.org/3/library/re.html](https://docs.python.org/3/library/re.html)
- Jurafsky, D., & Martin, J. H. (2023). *Speech and Language Processing* (3rd ed. draft). Chapter 2: Regular Expressions.
- NLTK: [https://www.nltk.org/](https://www.nltk.org/)

---